# Stage 8: Descriptive Outlier Detection Analysis
**Project**: Cyber Crime Analytics for National Security  
**Data Source**: National Crime Records Bureau (NCRB) 2023 Master Dataset (`master_state_2023.csv`) & EDA Matrix (`eda_state_feature_matrix.csv`)  
**Methodological Position**: Descriptive Statistical Identification of Extreme State/UT Observations ($N = 36$)

---

## 1. Objective & Analytical Scope

### Analytical Purpose:
The objective of this stage is to identify and characterize **statistically extreme State/UT observations** relative to the distribution of cybercrime volume, category concentrations, and motive shares in the validated 2023 NCRB dataset.

### Critical Methodological Guardrails:
1. **Statistical, Not Causal or Value Judgments**: This is an exploratory and descriptive analysis. Outliers are **statistical anomalies relative to the observed cross-sectional distribution**. They do **not** represent criminality rankings, risk scores, or proof that a state is "more dangerous".
2. **Plausible High-Volume Observations vs. Data Errors**: Extreme counts in major population/technology jurisdictions (e.g., Karnataka, Telangana, Uttar Pradesh, Maharashtra) represent genuine high reporting volume, **not data entry errors**.
3. **Small-Denominator Caution**: In small Union Territories with tiny total case counts ($N \le 10$), extreme motive proportions (e.g., 100% sexual exploitation motive in Lakshadweep on $N=1$) are driven by small denominators, **not high crime volume**.
4. **Dual Perspective (Univariate & Multivariate)**: We evaluate both dimension-specific statistical extremity (Tukey IQR fences) and joint multi-dimensional isolation (Isolation Forest on log-transformed counts + shares).


In [1]:
# Setup environment, paths, and imports
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Stage 8 outlier detection module routines
from src.outlier_detection import (
    load_and_prepare_outlier_data,
    compute_univariate_iqr_outliers,
    compute_multivariate_isolation_forest,
    build_state_outlier_summary,
    plot_outlier_iqr_boxplots,
    plot_outlier_flags_by_feature,
    plot_outlier_state_summary,
    plot_outlier_multivariate_projection,
    export_outlier_outputs,
    VOLUME_FEATURES,
    SHARE_FEATURES,
    ALL_OUTLIER_FEATURES,
    FEATURE_DISPLAY_NAMES
)

print(f"Working Directory: {project_root}")
print("Stage 8 Outlier Detection module loaded successfully.")


Working Directory: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics
Stage 8 Outlier Detection module loaded successfully.


---
## Section B — Feature Selection & Distribution Inspection

We analyze 14 non-redundant, domain-specific indicators spanning two analytical dimensions:
1. **Volume Dimensions (10 Count Features)**: `total_cases`, `it_act_cases`, `ipc_cases`, `motive_fraud`, `motive_extortion`, `motive_sexual_exploitation`, `sec66d_cheating_personation`, `sec66c_identity_theft`, `women_cases_total`, `child_cases_total`.
2. **Composition Dimensions (4 Share Features)**: `it_act_share`, `fraud_motive_share`, `extortion_motive_share`, `sexual_exploitation_motive_share`.


In [2]:
# Load 2023 dataset and prepare analytical features
eda_path = project_root / 'outputs' / 'tables' / 'eda_state_feature_matrix.csv'
master_path = project_root / 'data' / 'processed' / 'master_state_2023.csv'
cluster_path = project_root / 'outputs' / 'tables' / 'cluster_assignments_2023.csv'

features_df, metadata = load_and_prepare_outlier_data(eda_path, master_path, cluster_path)
print(f"Prepared Outlier Feature Matrix: {features_df.shape[0]} States/UTs x {len(ALL_OUTLIER_FEATURES)} Features")
display(features_df.head())


Prepared Outlier Feature Matrix: 36 States/UTs x 14 Features


,state_name,total_cases,it_act_cases,ipc_cases,motive_fraud,motive_extortion,motive_sexual_exploitation,sec66d_cheating_personation,sec66c_identity_theft,women_cases_total,child_cases_total,it_act_share,fraud_motive_share,extortion_motive_share,sexual_exploitation_motive_share
0,Andhra Pradesh,2341,362,1978,1334,126,239,162,65,559,83,0.154635,0.569842,0.053823,0.102093
1,Arunachal Pradesh,24,17,7,20,0,0,7,8,4,0,0.708333,0.833333,0.000000,0.000000
2,Assam,909,847,62,138,132,59,153,93,469,40,0.931793,0.151815,0.145215,0.064906
3,Bihar,4450,397,4052,3422,159,93,112,130,644,10,0.089213,0.768989,0.035730,0.020899
4,Chhattisgarh,473,310,111,49,3,136,6,3,242,193,0.655391,0.103594,0.006342,0.287526


---
## Section C — Primary Univariate Outlier Methodology (Tukey IQR Fences)

### Tukey IQR Fence Definition:
For each feature $X$:
- $Q_1 = 25\text{th percentile}$, $Q_3 = 75\text{th percentile}$
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Fence} = Q_1 - 1.5 \times \text{IQR}$
- $\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$

Observations with $x_i > \text{Upper Fence}$ are flagged as **High Outliers**; observations with $x_i < \text{Lower Fence}$ are flagged as **Low Outliers**.


In [3]:
# Compute univariate IQR statistics and identify fence violations
feature_stats_df, univariate_outliers_df = compute_univariate_iqr_outliers(features_df, ALL_OUTLIER_FEATURES)

print("--- Feature Distribution & Tukey IQR Fences Summary ---")
display(feature_stats_df[['feature_display', 'feature_type', 'q1', 'median', 'q3', 'iqr', 'upper_fence', 'skewness', 'total_outlier_count', 'flagged_states']])

print(f"\n--- Detailed Univariate Fence Violations (Total Occurrences: {len(univariate_outliers_df)}) ---")
display(univariate_outliers_df.head(15))


--- Feature Distribution & Tukey IQR Fences Summary ---

--- Detailed Univariate Fence Violations (Total Occurrences: 52) ---


,feature_display,feature_type,q1,median,q3,iqr,upper_fence,skewness,total_outlier_count,flagged_states
0,Total Cybercrime Cases,Volume (Count),34.7500,440.0000,2342.7500,2308.0000,5804.7500,2.9612,4,"Karnataka, Maharashtra, Telangana, Uttar Pradesh"
1,IT Act Cases,Volume (Count),30.7500,193.5000,568.7500,538.0000,1375.7500,4.7515,4,"Karnataka, Rajasthan, Tamil Nadu, Uttar Pradesh"
2,IPC Cases,Volume (Count),1.7500,69.5000,755.0000,753.2500,1884.8750,4.5071,6,"Andhra Pradesh, Bihar, Kerala, Maharashtra, Tamil Nadu, Telangana"
3,Fraud Motive Cases,Volume (Count),19.2500,119.5000,1223.7500,1204.5000,3030.5000,3.4948,6,"Bihar, Karnataka, Maharashtra, Tamil Nadu, Telangana, Uttar Pradesh"
4,Extortion Motive Cases,Volume (Count),1.0000,9.5000,123.7500,122.7500,307.8750,3.4807,3,"Karnataka, Kerala, Uttar Pradesh"
5,Sexual Exploitation Motive Cases,Volume (Count),8.7500,30.5000,121.7500,113.0000,291.2500,1.9343,5,"Karnataka, Kerala, Maharashtra, Rajasthan, Uttar Pradesh"
6,Sec. 66D Personation Cheating,Volume (Count),1.7500,37.5000,159.5000,157.7500,396.1250,5.7481,5,"Karnataka, Kerala, Rajasthan, Tamil Nadu, Uttar Pradesh"
7,Sec. 66C Identity Theft,Volume (Count),2.7500,8.0000,80.2500,77.5000,196.5000,5.0870,5,"Gujarat, Jharkhand, Karnataka, Maharashtra, Uttar Pradesh"
8,Cybercrimes Against Women,Volume (Count),13.0000,131.0000,530.5000,517.5000,1306.7500,4.4622,4,"Karnataka, Maharashtra, Telangana, Uttar Pradesh"
9,Cybercrimes Against Children,Volume (Count),2.5000,12.0000,39.2500,36.7500,94.3750,2.7680,4,"Chhattisgarh, Karnataka, Kerala, Rajasthan"


,state_name,feature,feature_type,observed_value,q1,q3,iqr,lower_fence,upper_fence,direction,distance_from_fence,total_cases_denominator
0,Kerala,child_cases_total,Volume (Count),443.0000,2.5000,39.2500,36.7500,-52.6250,94.3750,High,348.6250,3295
1,Karnataka,child_cases_total,Volume (Count),363.0000,2.5000,39.2500,36.7500,-52.6250,94.3750,High,268.6250,21889
2,Rajasthan,child_cases_total,Volume (Count),306.0000,2.5000,39.2500,36.7500,-52.6250,94.3750,High,211.6250,2435
3,Chhattisgarh,child_cases_total,Volume (Count),193.0000,2.5000,39.2500,36.7500,-52.6250,94.3750,High,98.6250,473
4,Kerala,extortion_motive_share,Composition (Share),0.1791,0.0067,0.0561,0.0495,-0.0675,0.1303,High,0.0487,3295
5,Assam,extortion_motive_share,Composition (Share),0.1452,0.0067,0.0561,0.0495,-0.0675,0.1303,High,0.0149,909
6,Uttarakhand,extortion_motive_share,Composition (Share),0.1417,0.0067,0.0561,0.0495,-0.0675,0.1303,High,0.0114,494
7,Chandigarh,extortion_motive_share,Composition (Share),0.1304,0.0067,0.0561,0.0495,-0.0675,0.1303,High,0.0001,23
8,Telangana,ipc_cases,Volume (Count),17787.0000,1.7500,755.0000,753.2500,-1128.1250,1884.8750,High,15902.1250,18236
9,Maharashtra,ipc_cases,Volume (Count),7131.0000,1.7500,755.0000,753.2500,-1128.1250,1884.8750,High,5246.1250,8103


---
## Section D — Multivariate Outlier Analysis (Isolation Forest)

To evaluate joint multi-dimensional extremity without raw volume scale domination:
1. Count features are transformed using $\log(1 + y)$ to stabilize variance and mitigate extreme right-skew.
2. Transformed counts and composition shares are standardized with `StandardScaler`.
3. An **Isolation Forest** ensemble ($contamination = 0.15, \text{random\_state} = 42$) isolates anomalous multi-dimensional profile combinations.


In [4]:
# Fit Isolation Forest and compute anomaly scores
multivariate_df, iso_model, X_scaled = compute_multivariate_isolation_forest(features_df, contamination=0.15, random_state=42)

print("--- Isolation Forest Multivariate Outliers (Top 10 Ranked by Anomaly Score) ---")
display(multivariate_df.head(10))


--- Isolation Forest Multivariate Outliers (Top 10 Ranked by Anomaly Score) ---


,state_name,total_cases,anomaly_score,isolation_forest_outlier,anomaly_rank
0,Karnataka,21889,-0.1139,Yes,1
1,Ladakh,1,-0.0505,Yes,2
2,Lakshadweep,1,-0.0229,Yes,3
3,Kerala,3295,-0.0184,Yes,4
4,Dadra and Nagar Haveli and Daman and Diu,6,-0.0022,Yes,5
5,Uttar Pradesh,10794,-0.0007,Yes,6
6,Odisha,2348,0.0021,No,7
7,Jharkhand,1079,0.0038,No,8
8,Sikkim,12,0.0135,No,9
9,Chandigarh,23,0.0173,No,10


---
## Section E — State-Level Outlier Synthesis & Aggregation

We synthesize univariate and multivariate results into four descriptive classifications:
1. **`Both`**: Flagged by both univariate Tukey fences and multivariate Isolation Forest ($n = 5$).
2. **`Univariate outlier`**: Flagged along one or more single dimensions, but not isolated multivariately ($n = 12$).
3. **`Multivariate outlier`**: Flagged multivariately due to sparse joint profile combinations (Ladakh, $n = 1$).
4. **`No detected outlier`**: Within expected distribution ranges across all evaluated features ($n = 18$).


In [5]:
# Build unified state outlier summary table
summary_df = build_state_outlier_summary(features_df, univariate_outliers_df, multivariate_df, metadata)

print("--- Overall State/UT Outlier Classification Breakdown ---")
print(summary_df['outlier_classification'].value_counts())

print("\n--- State/UT Outlier Summary Table (All 36 Jurisdictions) ---")
display(summary_df[['state_name', 'total_cases', 'univariate_flags_count', 'isolation_forest_outlier',
                    'outlier_classification', 'small_denominator_flag', 'stage6_cluster_label',
                    'flagged_features']])


--- Overall State/UT Outlier Classification Breakdown ---
outlier_classification
No detected outlier     18
Univariate outlier      12
Both                     5
Multivariate outlier     1
Name: count, dtype: int64

--- State/UT Outlier Summary Table (All 36 Jurisdictions) ---


,state_name,total_cases,univariate_flags_count,isolation_forest_outlier,outlier_classification,small_denominator_flag,stage6_cluster_label,flagged_features
0,Karnataka,21889,9,Yes,Both,No,Higher IT Act Share / Higher Fraud Share Profile,"child_cases_total, it_act_cases, motive_extortion, motive_fraud, motive_sexual_exploitation, sec66c_identity_theft, sec66d_cheating_personation, total_cases, women_cases_total"
1,Uttar Pradesh,10794,8,Yes,Both,No,Higher Extortion Motive Share Profile,"it_act_cases, motive_extortion, motive_fraud, motive_sexual_exploitation, sec66c_identity_theft, sec66d_cheating_personation, total_cases, women_cases_total"
2,Kerala,3295,6,Yes,Both,No,Higher Extortion Motive Share Profile,"child_cases_total, extortion_motive_share, ipc_cases, motive_extortion, motive_sexual_exploitation, sec66d_cheating_personation"
3,Maharashtra,8103,6,No,Univariate outlier,No,Lower IT Act Share / Moderate Fraud Share Profile,"ipc_cases, motive_fraud, motive_sexual_exploitation, sec66c_identity_theft, total_cases, women_cases_total"
4,Telangana,18236,4,No,Univariate outlier,No,Lower IT Act Share / Moderate Fraud Share Profile,"ipc_cases, motive_fraud, total_cases, women_cases_total"
5,Tamil Nadu,4121,4,No,Univariate outlier,No,Higher IT Act Share / Higher Fraud Share Profile,"ipc_cases, it_act_cases, motive_fraud, sec66d_cheating_personation"
6,Rajasthan,2435,4,No,Univariate outlier,No,Lower IT Act Share / Moderate Fraud Share Profile,"child_cases_total, it_act_cases, motive_sexual_exploitation, sec66d_cheating_personation"
7,Bihar,4450,2,No,Univariate outlier,No,Lower IT Act Share / Moderate Fraud Share Profile,"ipc_cases, motive_fraud"
8,Lakshadweep,1,1,Yes,Both,Yes (N <= 10),High Sexual-Exploitation Share / Small-Denominator Profile,sexual_exploitation_motive_share
9,Dadra and Nagar Haveli and Daman and Diu,6,1,Yes,Both,Yes (N <= 10),High Sexual-Exploitation Share / Small-Denominator Profile,sexual_exploitation_motive_share


---
## Section F — Diagnostic Visualizations


In [6]:
# Figure 24: IQR Boxplots
fig_boxplots = plot_outlier_iqr_boxplots(
    features_df,
    save_path=str(project_root / 'outputs' / 'figures' / '24_outlier_iqr_boxplots.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [7]:
# Figure 25: Outlier Counts by Feature
fig_flags = plot_outlier_flags_by_feature(
    feature_stats_df,
    save_path=str(project_root / 'outputs' / 'figures' / '25_outlier_flags_by_feature.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [8]:
# Figure 26: State/UT Summary Chart
fig_state_summary = plot_outlier_state_summary(
    summary_df,
    save_path=str(project_root / 'outputs' / 'figures' / '26_outlier_state_summary.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [9]:
# Figure 27: Multivariate 2D PCA Outlier Projection
fig_multi_proj = plot_outlier_multivariate_projection(
    X_scaled,
    summary_df,
    save_path=str(project_root / 'outputs' / 'figures' / '27_outlier_multivariate_projection.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section G — Analytical Interpretation & Methodological Findings

### 1. High-Volume Scale Observations:
- **Karnataka (21,889 cases)**, **Telangana (18,236 cases)**, **Uttar Pradesh (10,794 cases)**, and **Maharashtra (8,103 cases)** consistently exceed the upper IQR fence ($5,804.75$ cases) for total cybercrime.
- These 4 State/UT observations exceed this upper fence, reflecting high reporting volume rather than recording errors.

### 2. Dimension-Specific Volume Outliers:
- **Extortion Motive Outliers**: Uttar Pradesh ($1,020$ cases, fence: $307.9$), Kerala ($590$ cases), and Karnataka ($437$ cases).
- **Identity Theft Outliers (Sec. 66C)**: Karnataka ($2,411$ cases), Uttar Pradesh ($486$ cases), Jharkhand ($479$ cases), Maharashtra ($474$ cases), and Gujarat ($254$ cases, fence: $196.5$).
- **Cybercrimes Against Children**: Kerala ($443$ cases, fence: $94.4$), Karnataka ($363$ cases), Rajasthan ($306$ cases), and Chhattisgarh ($193$ cases).

### 3. Small-Denominator Caution (Share Outliers):
- **Dadra and Nagar Haveli and Daman and Diu** ($83.33\%$ sexual exploitation share) and **Lakshadweep** ($100.00\%$ sexual exploitation share) exceed the upper Tukey fence ($44.52\%$).
- **Critical Analytical Context**: These proportions are driven entirely by **tiny denominators** ($N = 6$ total cases and $N = 1$ total case respectively), **not high absolute crime volume**.


In [10]:
# Export Tables for Downstream Reporting & Power BI
exported_paths = export_outlier_outputs(
    feature_stats_df=feature_stats_df,
    univariate_outliers_df=univariate_outliers_df,
    multivariate_df=multivariate_df,
    summary_df=summary_df,
    output_dir=str(project_root / 'outputs' / 'tables')
)

print("Exported Stage 8 Outlier Output Files:")
for k, v in exported_paths.items():
    p = Path(v)
    print(f"  - {k:<25}: {p.name} ({p.stat().st_size:,} bytes)")


Exported Stage 8 Outlier Output Files:
  - feature_statistics       : outlier_feature_statistics.csv (2,727 bytes)
  - univariate_results       : outlier_univariate_results.csv (5,620 bytes)
  - multivariate_results     : outlier_multivariate_results.csv (1,137 bytes)
  - state_summary            : outlier_state_summary.csv (5,224 bytes)


---
## Section H — Methodological Limitations & Analytical Boundaries

### Explicit Constraints:
1. **Descriptive, Non-Causal Nature**: Outlier detection identifies observations that are statistically unusual relative to the observed distribution. It does not explain causal socio-economic mechanisms or assess criminality.
2. **Cross-Sectional Scope ($N = 36$)**: Outliers are defined relative to the 36 aggregate State/UT observations in the 2023 NCRB dataset.
3. **Reporting & Registration Practices**: Variations in observed volume may reflect differences in police registration practices, specialized cyber cells, and public reporting portals across jurisdictions.
